
# Physics-Informed Reinforcement Learning for PBF Toolpath Planning

This notebook implements the paper methodology as a research reproduction scaffold:

1. Procedural target geometries
2. A geometric toolpath environment
3. A finite-element thermal environment
4. DQN and PPO agents
5. A Zigzag reference policy
6. Evaluation on held-out shapes

The implementation is intentionally limited to the paper's simplified 2-D, single-layer benchmark.



## 1. Install dependencies

Run this cell once in a fresh environment. Install the PyTorch build appropriate for your CPU/CUDA configuration when necessary.


In [ ]:

# Uncomment in a fresh environment:
# %pip install numpy scipy matplotlib Pillow gymnasium stable-baselines3 torch


In [ ]:

from pathlib import Path
import sys

PROJECT_DIR = Path(".").resolve()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import numpy as np
import matplotlib.pyplot as plt

from pbf_rl_toolpath import (
    ProceduralMaskGenerator,
    GeometricPBFEnv,
    ThermalPBFEnv,
    ThermalParameters,
    run_zigzag_episode,
    plot_episode,
    train_sb3,
    evaluate_sb3_model,
)



## 2. Procedural geometry generation

The training, validation, and test distributions are deliberately separated:

- training: letters
- validation: symbols
- testing: geometric shapes


In [ ]:

generator = ProceduralMaskGenerator(n=10)
rng = np.random.default_rng(4)

fig, axes = plt.subplots(1, 3, figsize=(8, 3))
for ax, split in zip(axes, ["train", "validation", "test"]):
    sample = generator.sample(split, rng)
    ax.imshow(sample.mask, cmap="gray_r")
    ax.set_title(f"{split}: {sample.name}")
    ax.axis("off")
plt.tight_layout()



## 3. Geometric environment

Observation channels:

1. target mask
2. cumulative processed phase
3. source position and on/off state

Actions:

- 0: up
- 1: down
- 2: left
- 3: right
- 4: toggle source


In [ ]:

geom_env = GeometricPBFEnv(
    n=10,
    split="test",
    seed=10,
    fixed_mask_name="triangle",
)
obs, info = geom_env.reset()
print("Observation shape:", obs.shape)
print(info)

for _ in range(40):
    action = geom_env.action_space.sample()
    obs, reward, terminated, truncated, info = geom_env.step(action)
    if terminated or truncated:
        break

plot_episode(geom_env, "Random policy: geometric environment")
plt.show()



## 4. Thermal finite-element environment

The thermal state is advanced after every action using a structured bilinear Q4 finite-element discretization with backward-Euler time integration.

The observation contains:

1. target mask
2. source position/state
3. normalized temperature field
4. cumulative phase map


In [ ]:

thermal_env = ThermalPBFEnv(
    n=10,
    split="test",
    seed=12,
    fixed_mask_name="ring",
)
obs, info = thermal_env.reset()

# Turn on the source, then move through a short path.
actions = [4, 3, 3, 1, 1, 2, 2, 0, 0]
for action in actions:
    obs, reward, terminated, truncated, info = thermal_env.step(action)
    if terminated or truncated:
        break

print("Peak temperature:", info.get("peak_temperature"))
print("Coverage:", info["coverage"])
plot_episode(thermal_env, "Thermal environment")
plt.show()



## 5. Zigzag reference policy

The reference policy traverses the complete grid row by row and activates the source only over unprocessed target cells.


In [ ]:

zig_env = ThermalPBFEnv(
    n=10,
    split="test",
    seed=20,
    fixed_mask_name="square",
)
zigzag_metrics = run_zigzag_episode(zig_env)
print(zigzag_metrics)
plot_episode(zig_env, "Zigzag baseline")
plt.show()



## 6. Train PPO or DQN

Training directly against the finite-element environment is computationally expensive.

Start with the geometric environment and a short run. Then transfer the workflow to the thermal environment.

The paper-scale experiment requires substantially more timesteps and multiple independent seeds.


In [ ]:

# Short geometric PPO demonstration:
# model = train_sb3(
#     algorithm="ppo",
#     env_kind="geometric",
#     total_timesteps=100_000,
#     n=10,
#     seed=0,
#     output_path="ppo_geometric",
# )

# Thermal PPO:
# model = train_sb3(
#     algorithm="ppo",
#     env_kind="thermal",
#     total_timesteps=500_000,
#     n=10,
#     seed=0,
#     output_path="ppo_thermal",
# )

# DQN:
# model = train_sb3(
#     algorithm="dqn",
#     env_kind="geometric",
#     total_timesteps=100_000,
#     n=10,
#     seed=0,
#     output_path="dqn_geometric",
# )



## 7. Evaluate a trained model on held-out shapes


In [ ]:

# metrics = evaluate_sb3_model(
#     model_path="ppo_thermal",
#     algorithm="ppo",
#     env_kind="thermal",
#     n=10,
#     episodes=50,
# )
# metrics



## 8. Recommended research extensions

- Add laser-power modulation and scan speed as continuous actions.
- Add reward terms for peak temperature, temperature gradients, and cooling rate.
- Add temperature-dependent material properties and latent heat.
- Use a surrogate thermal model to accelerate training.
- Extend the environment to multi-layer 3-D paths.
- Introduce simulated sensor noise and in-situ thermal measurements for sim-to-real transfer.
